# RSNA Knee: offline inference (cross-slice attention on frozen DINOv2 features)

Inputs: competition data, `knee-inference` (package source + fold checkpoints), `knee-weights` (backbone).
Internet is not needed. Set `KNEE_DRY_RUN=5` to run on the first 5 test studies only.

In [ ]:
import os, sys, time, json
from pathlib import Path

T0 = time.time()
DRY_RUN = int(os.environ.get("KNEE_DRY_RUN", "0"))


def find_dir(marker, root="/kaggle/input", max_depth=6, skip=("train_series", "test_series")):
    """First folder under /kaggle/input containing `marker` (skips the big DICOM trees)."""
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        dirnames[:] = sorted(d for d in dirnames if d not in skip and depth < max_depth)
        if marker in filenames:
            return Path(dirpath)
    raise FileNotFoundError(f"{marker} not found under {root}")


BUNDLE = find_dir("knee_bundle.json")
WEIGHTS = find_dir("knee_weights.json")
print("bundle:", BUNDLE, json.loads((BUNDLE / "knee_bundle.json").read_text()))
print("weights:", WEIGHTS, sorted(p.name for p in WEIGHTS.iterdir()))
sys.path.insert(0, str(BUNDLE / "src"))

In [ ]:
from knee.config import load_config
from knee.infer.predict import run_inference, write_submission
from knee.utils import get_logger

cfg = load_config(overrides=[
    f"backbone.weights_path={WEIGHTS}",
    f"infer.checkpoints={BUNDLE / 'models'}",
    "infer.output=/kaggle/working/submission.csv",
    "preprocess.num_workers=4",
    "infer.tta_mirror=false",
])
log = get_logger("knee.infer", "/kaggle/working/inference.log")
sub = run_inference(cfg, log, dry_run=DRY_RUN)
write_submission(sub, "/kaggle/working/submission.csv", Path(cfg.paths.data_root) / "sample_submission.csv", log)
print(f"notebook wall time: {time.time() - T0:.1f}s")
sub.head()